# Remover duplicados de InChIKeys Manteniendo el mayor pKd


## 1. Importar Bibliotecas

In [27]:
import pandas as pd
import numpy as np
import os


## 2. Configurar los archivos

In [28]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:

df = pd.read_csv("/content/drive/MyDrive/FBDD_Denovo_VRC/smHDAC8_Prueba_all_1000_Druglike/Analisis de resultados/extracted_denovo.csv", sep=';')   # Archivo con InChIKeys y otras propiedades
df



,SMILES,Name,InChI,Formula,MW,LogP,pKd,Structure,Synthesize,Chemical
0,OC1C=CC(=CC=1C/C=C\CN)Cl,1/000000061,InChI=1S/C10H12ClNO/c11-9-4-5-10(13)8(7-9)3-1-...,C10H12NOCl,197,2.11,3.56,-4.17,2.22,0
1,OC1=CC=C(C(=C1)CC[C@@H](CCF)O)Cl,1/000000101,InChI=1S/C11H14ClFO2/c12-11-4-3-10(15)7-8(11)1...,C11H14O2FCl,232,2.97,3.22,-7.78,2.22,-150
2,OC1C=CC(=C(C=1)C1=NOC(=C1)C(=O)NC)Cl,1/000000086,InChI=1S/C11H9ClN2O3/c1-13-11(16)10-5-9(14-17-...,C11H9N2O3Cl,252,2.24,3.39,-11.07,17.50,-180
3,[NH2]=C1[N]C(=NN1)[C@@H](CCC1=CC(=O)NC1=O)Br,1/000000053,InChI=1S/C9H10BrN5O2/c10-5(7-13-9(11)15-14-7)2...,C9H10N5O2Br,300,-0.19,3.27,-1.16,2.22,-150
4,O[C@](C1=CNC(=C1)CNC)(C1=NN=CN1)C=C,1/000000033,"InChI=1S/C11H15N5O/c1-3-11(17,10-14-7-15-16-10...",C11H15N5O,233,0.93,3.29,-2.24,2.22,-170
...,...,...,...,...,...,...,...,...,...,...
1007,OCNCCCC1=C(Cl)C=CC(=C1)Cl,6/000000010,InChI=1S/C10H13Cl2NO/c11-9-3-4-10(12)8(6-9)2-1...,C10H13NOCl2,233,2.72,3.46,-0.23,2.22,0
1008,OC(=O)C1=NN=C(N1)CCC1=CC=C(N1)CO,6/000000031,InChI=1S/C10H12N4O3/c15-5-7-2-1-6(11-7)3-4-8-1...,C10H12N4O3,236,0.62,3.67,-5.85,2.22,-60
1009,OCC1=NN=C2N1C[C@@H](NC2)/C=C/CN,6/000000081,InChI=1S/C9H15N5O/c10-3-1-2-7-5-14-8(4-11-7)12...,C9H15N5O,209,-0.21,3.32,-4.92,16.40,-190
1010,OC(=O)C1=NN=C(N1)CCC1=CC=C(N1)CO,6/000000032,InChI=1S/C10H12N4O3/c15-5-7-2-1-6(11-7)3-4-8-1...,C10H12N4O3,236,0.62,3.59,-7.00,2.22,-60


In [30]:
input_filename = 'extracted_denovo.csv'
output_filename = 'dados_sem_duplicatas.csv'


## 3. Identificar duplicados de InChIKeys

In [31]:
# Verificar cuales InChIKeys estan duplicados
duplicados = df[df.duplicated(subset='InChI', keep=False)]
print(f"\nInChIs duplicados: {duplicados['InChI'].nunique()} únicos")
print(f"Total de lineas duplicadas: {len(duplicados)}")
print("\nMoléculas con InChIs duplicados:")
print(duplicados.sort_values('InChI'))


InChIs duplicados: 194 únicos
Total de lineas duplicadas: 906

Moléculas con InChIs duplicados:
                               SMILES         Name  \
198  OCC(=O)OCC1C=C(Cl)C=C(C=1)C(=O)N  2/000000124   
111  OCC(=O)OCC1C=C(Cl)C=C(C=1)C(=O)N  2/000000121   
112  OCC(=O)OCC1C=C(Cl)C=C(C=1)C(=O)N  2/000000123   
114  OCC(=O)OCC1C=C(Cl)C=C(C=1)C(=O)N  2/000000120   
211  OCC(=O)OCC1C=C(Cl)C=C(C=1)C(=O)N  2/000000122   
..                                ...          ...   
126       OC1C=CC(=C(C=1)I)CC(=O)NCBr  2/000000042   
117       OC1C=CC(=C(C=1)I)CC(=O)NCBr  2/000000075   
131       OC1C=CC(=C(C=1)I)CC(=O)NCBr  2/000000011   
991  OC(=O)[C@H](C1=CC(N)=CC=C1F)OCCl  6/000000090   
929  OC(=O)[C@H](C1=CC(N)=CC=C1F)OCCl  6/000000091   

                                                 InChI      Formula   MW  \
198  InChI=1S/C10H10ClNO4/c11-8-2-6(5-16-9(14)4-13)...  C10H10NO4Cl  243   
111  InChI=1S/C10H10ClNO4/c11-8-2-6(5-16-9(14)4-13)...  C10H10NO4Cl  243   
112  InChI=1S/C10H10ClNO4/

## 4. Mantener las Moléculas con mayor pKd

In [32]:
# Ordena por pKd descendente y mantener la primera ocurrencia de cada InChI
df_best = df.loc[df.groupby("InChI")["pKd"].idxmax()]
df_final = df_best.sort_values('pKd', ascending=False)
df_final

,SMILES,Name,InChI,Formula,MW,LogP,pKd,Structure,Synthesize,Chemical
269,OC(=O)C1C=CC=C2C=1N/C(=C/CC(=O)NCC1=CC=CC3=C1C...,3/000000487,InChI=1S/C21H17N3O5/c25-16(22-10-12-4-2-5-14-1...,C21H17N3O5,391,1.84,5.39,-8.34,6.45,-60
572,ClC1=CC=C(C=C1)COCCNC(=O)C/C=C\1/NC2C(C1)=CC=C...,3/000000194,InChI=1S/C22H25ClN2O2/c1-2-16-3-6-18-14-20(25-...,C22H25N2O2Cl,384,4.17,4.95,-7.42,6.45,-100
512,C1C(OCC2=NC=CC(=C2)C(=O)N)=CC(=CC=1)N1C(=CCNC1...,3/000000246,InChI=1S/C18H18N4O4/c19-17(23)12-4-7-21-13(8-1...,C18H17N4O4,353,0.87,4.92,-2.59,22.40,-180
595,C1=NC2C(C(=O)O1)=CC=CC=2NC(=O)C/C=C\1/NC2C(C1)...,3/000000066,InChI=1S/C21H19N3O3/c1-2-13-6-7-14-11-15(23-18...,C21H19N3O3,361,2.80,4.90,-6.57,6.45,-100
381,OC(=O)C1C=CC=C2C=1N/C(=C/CC(=O)NC[C@@H](CO)F)/C2,3/000000466,InChI=1S/C15H17FN2O4/c16-10(8-19)7-17-13(20)5-...,C15H17N2O4F,308,0.96,4.90,-2.45,4.94,-190
...,...,...,...,...,...,...,...,...,...,...
979,C(=C\CN)/C1=CC=C2C(=C1)C(=O)C=CN2,6/000000083,InChI=1S/C12H12N2O/c13-6-1-2-9-3-4-11-10(8-9)1...,C12H12N2O,200,1.51,3.01,-9.66,3.58,-80
775,OC(=O)C/C=C/C1C=CC=C(C=1Cl)O,4/000000101,InChI=1S/C10H9ClO3/c11-10-7(3-1-5-8(10)12)4-2-...,C10H9O3Cl,212,2.27,3.01,-2.70,2.22,0
897,OC(=O)/C=C/CCC1=NC2=C(N1)N=CC=C2,4/000000122,InChI=1S/C11H11N3O2/c15-10(16)6-2-1-5-9-13-8-4...,C11H11N3O2,217,1.00,3.01,-10.12,4.81,-100
178,OC1=CC=CC(=C1N1C=C[N]C1=[NH2])I,2/000000087,InChI=1S/C9H8IN3O/c10-6-2-1-3-7(14)8(6)13-5-4-...,C9H8N3OI,301,1.93,3.01,-6.51,13.40,-120


## 5. Verificar Resultado Final

Confirmar que se han eliminado los InChIKeys duplicados e visualizar las moléculas.

In [33]:
# Verificar se ainda há duplicatas
print("Verificación del DataFrame final:")
print(f"Total de InChIs únicos: {df_final['InChI'].nunique()}")
print(f"Total de lineas: {len(df_final)}")
print(f"Hay moléculas duplicadas? {df_final['InChI'].duplicated().any()}")

# comparación con DataFrame original
print(f"DataFrame original:")
print(f"DataFrame original: {len(df)} lineas")
print(f"DataFrame final: {len(df_final)} lineas")
print(f"lineas removidas: {len(df) - len(df_final)}")

Verificación del DataFrame final:
Total de InChIs únicos: 300
Total de lineas: 300
Hay moléculas duplicadas? False
DataFrame original:
DataFrame original: 1012 lineas
DataFrame final: 300 lineas
lineas removidas: 712


In [35]:
# Guardar DataFrame final
df_final.to_csv("/content/drive/MyDrive/FBDD_Denovo_VRC/smHDAC8_Prueba_all_1000_Druglike/Analisis de resultados/df_filtered_denovo.csv", index=False)
